In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
DOWNLOADER WAVEFORM 3-KOMPONEN DARI KATALOG BMKG + QC SNR
Untuk keperluan benchmarking MCU-Quake
"""

import os
import time
import numpy as np
import pandas as pd
from obspy import UTCDateTime, Stream
from obspy.clients.fdsn import Client
from tqdm import tqdm

# =============================================
# 1. KONFIGURASI (SESUAIKAN DENGAN NEED ANDA)
# =============================================

CATALOG_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog.csv"
OUTPUT_DIR = "/Volumes/Extreme SSD/unduhan_juni_bmkg_usgs"

MIN_MAGNITUDE = 4.5
MAX_RADIUS_DEG = 4.0
SNR_THRESHOLD = 4.0
NOISE_WINDOW = 10
SIGNAL_WINDOW = 20
TIME_BEFORE = 60
TIME_AFTER = 180

# =============================================
# 2. FUNGSI MEMBACA KATALOG BMKG (CSV)
# =============================================

def read_bmkg_catalog(csv_path):
    """
    Membaca file CSV BMKG dengan kolom standar:
    Date, Time (UTC), Latitude, Longitude, Magnitude, Depth (km)
    """
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    
    print("="*70)
    print("📂  STRUKTUR KATALOG")
    print("="*70)
    print(f"Kolom yang ditemukan: {df.columns.tolist()}")
    print("\nContoh 3 baris pertama:")
    print(df.head(3))
    print("="*70)
    
    events = []
    
    for idx, row in df.iterrows():
        try:
            # Gabungkan Date dan Time
            date_str = str(row['Date']).strip()
            time_str = str(row['Time (UTC)']).strip()
            if date_str == 'nan' or time_str == 'nan' or not date_str or not time_str:
                continue
            
            # Gunakan pandas untuk parsing tanggal (lebih tangguh)
            datetime_str = f"{date_str} {time_str}"
            # Contoh: 09-Jan-1998 09:45:43 -> pandas bisa parse dengan format
            dt_pd = pd.to_datetime(datetime_str, format='%d-%b-%Y %H:%M:%S')
            # Konversi ke UTCDateTime
            dt = UTCDateTime(dt_pd)
            
            lat = float(row['Latitude'])
            lon = float(row['Longitude'])
            mag = float(row['Magnitude']) if not pd.isna(row['Magnitude']) else None
            if mag is None:
                continue
            depth = float(row['Depth (km)']) if not pd.isna(row['Depth (km)']) else 0.0
            
            events.append({
                'time': dt,
                'lat': lat,
                'lon': lon,
                'mag': mag,
                'depth': depth,
                'id': dt.strftime("%Y%m%d_%H%M%S")
            })
        except Exception as e:
            # Tampilkan error untuk debugging (opsional)
            # print(f"⚠️  Baris {idx} gagal: {e}")
            continue
    
    print(f"✅  Total {len(events)} event berhasil dibaca dari katalog.\n")
    return events

# =============================================
# 3. FUNGSI MENGHITUNG SNR
# =============================================

def calculate_snr(trace, event_time, noise_window=10, signal_window=20):
    stats = trace.stats
    sr = stats.sampling_rate
    
    noise_start = event_time - noise_window
    noise_end = event_time
    signal_start = event_time
    signal_end = event_time + signal_window
    
    t0 = stats.starttime
    try:
        n_start = int((noise_start - t0) * sr)
        n_end = int((noise_end - t0) * sr)
        s_start = int((signal_start - t0) * sr)
        s_end = int((signal_end - t0) * sr)
    except:
        return None
    
    data = trace.data
    if n_start < 0 or n_end > len(data) or s_start < 0 or s_end > len(data):
        return None
    if n_end - n_start < 5 or s_end - s_start < 5:
        return None
    
    noise = data[n_start:n_end]
    signal = data[s_start:s_end]
    
    rms_noise = np.sqrt(np.mean(noise**2))
    rms_signal = np.sqrt(np.mean(signal**2))
    
    if rms_noise == 0:
        return None
    return rms_signal / rms_noise

# =============================================
# 4. FUNGSI MENCARI STASIUN TERDEKAT
# =============================================

def find_nearest_station(client, lat, lon, max_radius_deg=4.0):
    try:
        inv = client.get_stations(
            latitude=lat,
            longitude=lon,
            maxradius=max_radius_deg,
            level="channel",
            channel="BHZ"
        )
        if not inv:
            inv = client.get_stations(
                latitude=lat,
                longitude=lon,
                maxradius=max_radius_deg,
                level="channel",
                channel="HHZ"
            )
        if not inv:
            return None, None
        
        best_dist = float('inf')
        best_net = None
        best_sta = None
        
        for network in inv:
            for station in network:
                if station.latitude is None or station.longitude is None:
                    continue
                dist = ((station.latitude - lat)**2 + (station.longitude - lon)**2)**0.5
                if dist < best_dist:
                    best_dist = dist
                    best_net = network.code
                    best_sta = station.code
        return best_net, best_sta
    except Exception as e:
        print(f"⚠️  Error mencari stasiun: {e}")
        return None, None

# =============================================
# 5. FUNGSI DOWNLOAD 3 KOMPONEN + QC SNR
# =============================================

def download_3component_qc(client, event, output_dir):
    lat = event['lat']
    lon = event['lon']
    mag = event['mag']
    origin = event['time']
    event_id = event['id']
    
    network, station = find_nearest_station(client, lat, lon, MAX_RADIUS_DEG)
    if not network or not station:
        print(f"⚠️  Event {event_id} (M{mag:.1f}): Tidak ada stasiun dalam radius {MAX_RADIUS_DEG}°")
        return False
    
    starttime = origin - TIME_BEFORE
    endtime = origin + TIME_AFTER
    
    channel_sets = [
        ['BHZ', 'BHN', 'BHE'],
        ['HHZ', 'HHN', 'HHE']
    ]
    
    for ch_set in channel_sets:
        stream = Stream()
        all_ok = True
        for ch in ch_set:
            try:
                tr = client.get_waveforms(
                    network=network,
                    station=station,
                    location="",
                    channel=ch,
                    starttime=starttime,
                    endtime=endtime
                )
                if tr and len(tr) > 0:
                    stream += tr
                else:
                    all_ok = False
                    break
            except Exception as e:
                all_ok = False
                break
        
        if all_ok and len(stream) == 3:
            snr_list = []
            for tr in stream:
                snr = calculate_snr(tr, origin, NOISE_WINDOW, SIGNAL_WINDOW)
                if snr is not None:
                    snr_list.append(snr)
                else:
                    snr_list.append(0.0)
            
            if len(snr_list) == 3:
                avg_snr = np.mean(snr_list)
                print(f"📊  Event {event_id} (M{mag:.1f}) | {network}.{station} | SNR: Z={snr_list[0]:.2f}, N={snr_list[1]:.2f}, E={snr_list[2]:.2f} | Rata-rata={avg_snr:.2f}")
                
                if avg_snr >= SNR_THRESHOLD:
                    os.makedirs(output_dir, exist_ok=True)
                    filename = os.path.join(output_dir, f"{network}_{station}_{event_id}.mseed")
                    stream.write(filename, format="MSEED")
                    print(f"✅  LULUS QC → {filename}")
                    return True
                else:
                    print(f"❌  GAGAL QC (SNR rata-rata {avg_snr:.2f} < {SNR_THRESHOLD})")
                    return False
            else:
                print(f"⚠️  Gagal menghitung SNR untuk event {event_id}")
                return False
        else:
            continue
    
    print(f"❌  Event {event_id}: Tidak ada data 3 komponen lengkap untuk stasiun {network}.{station}")
    return False

# =============================================
# 6. MAIN PROGRAM
# =============================================

if __name__ == "__main__":
    print("="*70)
    print("🚀  DOWNLOAD WAVEFORM 3-KOMPONEN DARI KATALOG BMKG + QC SNR")
    print("="*70)
    print(f"📂  Katalog: {CATALOG_PATH}")
    print(f"📁  Output : {OUTPUT_DIR}")
    print("="*70)
    
    events = read_bmkg_catalog(CATALOG_PATH)
    if not events:
        print("❌  Tidak ada event yang terbaca. Periksa format CSV Anda.")
        # Tampilkan beberapa baris untuk debugging
        df = pd.read_csv(CATALOG_PATH)
        print("\n📄  ️Isi file CSV (10 baris pertama):")
        print(df.head(10))
        exit()
    
    filtered_events = [e for e in events if e['mag'] >= MIN_MAGNITUDE]
    print(f"🎯  {len(filtered_events)} event dengan magnitudo ≥ {MIN_MAGNITUDE} akan diproses.")
    
    if len(filtered_events) == 0:
        mags = [e['mag'] for e in events]
        print(f"📊  Statistik magnitudo di katalog: min={min(mags):.2f}, max={max(mags):.2f}, rata-rata={np.mean(mags):.2f}")
        print("💡  Coba turunkan nilai MIN_MAGNITUDE jika ingin memproses lebih banyak gempa.")
        exit()
    
    client = Client("IRIS")
    print(f"🌐  Terhubung ke IRIS DMC...")
    
    successful = 0
    total = len(filtered_events)
    
    for i, event in enumerate(tqdm(filtered_events, desc="Memproses event"), 1):
        print(f"\n--- Event {i}/{total}: {event['id']} (M{event['mag']:.1f}, {event['lat']:.2f}°, {event['lon']:.2f}°) ---")
        if download_3component_qc(client, event, OUTPUT_DIR):
            successful += 1
        time.sleep(0.5)
    
    print("\n" + "="*70)
    print("✨  SELESAI!")
    print("="*70)
    print(f"✅  Berhasil mengunduh: {successful} dari {total} event")
    print(f"❌  Gagal/lewat      : {total - successful} event")
    print(f"📁  Data tersimpan di: {OUTPUT_DIR}")
    print("="*70)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CATALOG_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog.csv"

# Baca CSV
df = pd.read_csv(CATALOG_PATH)
df.columns = df.columns.str.strip()

# Ambil kolom magnitudo
mag_series = df['Magnitude'].dropna()

print("="*60)
print("📊  ANALISIS DISTRIBUSI MAGNITUDO KATALOG")
print("="*60)

# Statistik dasar
print(f"Total event (dengan magnitudo): {len(mag_series)}")
print(f"Magnitudo minimum           : {mag_series.min():.2f}")
print(f"Magnitudo maksimum           : {mag_series.max():.2f}")
print(f"Rata-rata                    : {mag_series.mean():.2f}")
print(f"Median                       : {mag_series.median():.2f}")
print(f"Standar deviasi              : {mag_series.std():.2f}")

# Hitung jumlah event per rentang magnitudo
bins = [4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0]
labels = ['4.0-4.5', '4.5-5.0', '5.0-5.5', '5.5-6.0', '6.0-6.5', '6.5-7.0', '7.0-7.5', '7.5-8.0']
mag_cut = pd.cut(mag_series, bins=bins, labels=labels, right=False)
counts = mag_cut.value_counts().sort_index()

print("\n📈  Distribusi per rentang magnitudo:")
for label, count in counts.items():
    pct = (count / len(mag_series)) * 100
    print(f"   {label}  : {count:>6} event  ({pct:>5.2f}%)")

# Rekomendasi otomatis
print("\n" + "="*60)
print("💡  REKOMENDASI AMBANG BATAS UNDUHAN")
print("="*60)

# Ambil persentil 90 (hanya 10% gempa terbesar)
threshold_90 = mag_series.quantile(0.90)
threshold_95 = mag_series.quantile(0.95)
threshold_99 = mag_series.quantile(0.99)

print(f"📌  Magnitudo persentil 90%  : {threshold_90:.2f}  (hanya 10% gempa terbesar)")
print(f"📌  Magnitudo persentil 95%  : {threshold_95:.2f}  (hanya 5% gempa terbesar)")
print(f"📌  Magnitudo persentil 99%  : {threshold_99:.2f}  (hanya 1% gempa terbesar)")

print("\n🎯  Saran untuk MCU-Quake:")
print(f"   - Jika target > 100 event, gunakan MIN_MAGNITUDE = {threshold_90:.1f} (~{int(len(mag_series)*0.10)} event)")
print(f"   - Jika target 20-50 event, gunakan MIN_MAGNITUDE = {threshold_95:.1f} (~{int(len(mag_series)*0.05)} event)")
print(f"   - Jika target < 10 event, gunakan MIN_MAGNITUDE = {threshold_99:.1f} (~{int(len(mag_series)*0.01)} event)")

# --- Plot Histogram ---
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(mag_series, bins=30, edgecolor='black', alpha=0.7, color='skyblue')
plt.axvline(threshold_90, color='red', linestyle='--', label=f'Persentil 90% ({threshold_90:.1f})')
plt.axvline(threshold_95, color='orange', linestyle='--', label=f'Persentil 95% ({threshold_95:.1f})')
plt.axvline(threshold_99, color='green', linestyle='--', label=f'Persentil 99% ({threshold_99:.1f})')
plt.xlabel('Magnitudo')
plt.ylabel('Jumlah Event')
plt.title('Distribusi Magnitudo Katalog BMKG')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
# Plot kumulatif
sorted_mag = np.sort(mag_series)
cumulative = np.arange(1, len(sorted_mag)+1) / len(sorted_mag) * 100
plt.plot(sorted_mag, cumulative, 'b-', linewidth=2)
plt.axhline(90, color='red', linestyle='--', label='90%')
plt.axhline(95, color='orange', linestyle='--', label='95%')
plt.axhline(99, color='green', linestyle='--', label='99%')
plt.xlabel('Magnitudo')
plt.ylabel('Persentil Kumulatif (%)')
plt.title('Kurva Persentil Magnitudo')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('magnitude_distribution.png', dpi=150)
print("\n📁  Grafik disimpan sebagai 'magnitude_distribution.png'")
plt.show()

In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
PRE-SCAN KATALOG BMKG: Cocokkan dengan ketersediaan stasiun di IRIS dan GEOFON.
Tanpa mengunduh waveform.
"""

import os
import sys
import time
import pandas as pd
import numpy as np
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
from tqdm import tqdm
import logging

# =============================================
# 1. KONFIGURASI
# =============================================

CATALOG_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog.csv"
OUTPUT_CSV = "/Volumes/Extreme SSD/unduhan_juni_bmkg_usgs/hasilscan/bmkg_iris_geofon_feasible_events.csv"
OUTPUT_STATS = "bmkg_pre_scan_stats.txt"

MIN_MAGNITUDE = 3.0
YEAR_START = 2010
YEAR_END = 2024
MAX_RADIUS_DEG = 10.0       # radius pencarian stasiun

# Channel yang dicari (tambahkan alternatif)
CHANNELS = ['BHZ', 'HHZ', 'EHZ', 'HNZ']
# Lokasi wildcard
LOCATION = "*"

# Client
iris_client = Client("IRIS")
geofon_client = Client("GEOFON")

# Cache: (client_name, lat, lon, radius) -> (network, station, distance)
cache = {}

# =============================================
# 2. SETUP LOGGING
# =============================================

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# =============================================
# 3. BACA KATALOG BMKG
# =============================================

def read_bmkg_catalog(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    logger.info("="*70)
    logger.info("📂  MEMBACA KATALOG BMKG")
    logger.info("="*70)
    logger.info(f"Kolom: {df.columns.tolist()}")
    
    events = []
    for idx, row in df.iterrows():
        try:
            date_str = str(row['Date']).strip()
            time_str = str(row['Time (UTC)']).strip()
            if date_str == 'nan' or time_str == 'nan' or not date_str or not time_str:
                continue
            
            datetime_str = f"{date_str} {time_str}"
            dt_pd = pd.to_datetime(datetime_str, format='%d-%b-%Y %H:%M:%S')
            dt = UTCDateTime(dt_pd)
            
            if dt.year < YEAR_START or dt.year > YEAR_END:
                continue
            
            lat = float(row['Latitude'])
            lon = float(row['Longitude'])
            mag = float(row['Magnitude']) if not pd.isna(row['Magnitude']) else None
            if mag is None or mag < MIN_MAGNITUDE:
                continue
            
            events.append({
                'time': dt,
                'lat': lat,
                'lon': lon,
                'mag': mag,
                'id': dt.strftime("%Y%m%d_%H%M%S")
            })
        except Exception as e:
            continue
    
    logger.info(f"✅  {len(events)} event lolos filter (M≥{MIN_MAGNITUDE}, {YEAR_START}-{YEAR_END})")
    return events

# =============================================
# 4. CEK STASIUN DI SATU SERVER (DIPERBAIKI)
# =============================================

def find_station(client, client_name, lat, lon, origin_time, radius_deg=10.0):
    """
    Cari stasiun terdekat dengan channel yang diinginkan dalam radius.
    Return (network, station, distance) atau (None, None, None)
    """
    cache_key = (client_name, round(lat,2), round(lon,2), radius_deg)
    if cache_key in cache:
        return cache[cache_key]
    
    # Coba beberapa kali jika gagal (rate limit)
    for attempt in range(3):
        try:
            inventory = client.get_stations(
                latitude=lat,
                longitude=lon,
                maxradius=radius_deg,
                level="channel",
                location=LOCATION,
                channel=",".join(CHANNELS),
                # Rentang waktu lebih longgar: 1 hari sebelum dan sesudah
                starttime=origin_time - 86400,
                endtime=origin_time + 86400,
                limit=50
            )
            if inventory:
                break
        except Exception as e:
            if attempt == 2:
                cache[cache_key] = (None, None, None)
                return None, None, None
            time.sleep(2 ** attempt)  # exponential backoff
    
    if not inventory:
        cache[cache_key] = (None, None, None)
        return None, None, None
    
    stations = []
    for net in inventory:
        for sta in net:
            if sta.latitude is None or sta.longitude is None:
                continue
            dlat = sta.latitude - lat
            dlon = sta.longitude - lon
            dist = np.sqrt(dlat**2 + dlon**2)
            for ch in sta.channels:
                if ch.code in CHANNELS:
                    stations.append((net.code, sta.code, dist, ch.code))
                    break  # cukup satu channel per stasiun
    if not stations:
        cache[cache_key] = (None, None, None)
        return None, None, None
    
    # Urutkan berdasarkan jarak
    stations.sort(key=lambda x: x[2])
    best_net, best_sta, best_dist, _ = stations[0]
    cache[cache_key] = (best_net, best_sta, best_dist)
    return best_net, best_sta, best_dist

# =============================================
# 5. SCAN SEMUA EVENT
# =============================================

def scan_events(events, radius_deg=10.0):
    results = []
    stats = {
        'total': len(events),
        'iris_available': 0,
        'geofon_available': 0,
        'both_available': 0,
        'none_available': 0
    }
    
    logger.info("🔍  SCANNING ketersediaan stasiun...")
    
    for ev in tqdm(events, desc="Scanning", unit="event"):
        lat = ev['lat']; lon = ev['lon']; origin = ev['time']
        
        # Cek IRIS
        net_i, sta_i, dist_i = find_station(iris_client, 'IRIS', lat, lon, origin, radius_deg)
        # Cek GEOFON
        net_g, sta_g, dist_g = find_station(geofon_client, 'GEOFON', lat, lon, origin, radius_deg)
        
        iris_ok = net_i is not None
        geofon_ok = net_g is not None
        
        if iris_ok and geofon_ok:
            stats['both_available'] += 1
        elif iris_ok:
            stats['iris_available'] += 1
        elif geofon_ok:
            stats['geofon_available'] += 1
        else:
            stats['none_available'] += 1
        
        results.append({
            'event_id': ev['id'],
            'time': ev['time'].datetime,
            'lat': lat,
            'lon': lon,
            'mag': ev['mag'],
            'iris_available': iris_ok,
            'iris_network': net_i if iris_ok else '',
            'iris_station': sta_i if iris_ok else '',
            'iris_distance': round(dist_i, 3) if iris_ok else None,
            'geofon_available': geofon_ok,
            'geofon_network': net_g if geofon_ok else '',
            'geofon_station': sta_g if geofon_ok else '',
            'geofon_distance': round(dist_g, 3) if geofon_ok else None,
            'any_available': iris_ok or geofon_ok
        })
    
    return results, stats

# =============================================
# 6. SIMPAN HASIL
# =============================================

def save_results(results, stats, output_csv, output_stats):
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    logger.info(f"✅  Hasil scan disimpan ke: {output_csv}")
    
    with open(output_stats, 'w') as f:
        f.write("="*70 + "\n")
        f.write("📊  PRE-SCAN STATISTIK\n")
        f.write("="*70 + "\n")
        f.write(f"Total event              : {stats['total']}\n")
        f.write(f"IRIS saja               : {stats['iris_available']}\n")
        f.write(f"GEOFON saja             : {stats['geofon_available']}\n")
        f.write(f"Keduanya                : {stats['both_available']}\n")
        f.write(f"Tidak ada               : {stats['none_available']}\n")
        f.write(f"Minimal satu tersedia   : {stats['total'] - stats['none_available']}\n")
        f.write("="*70 + "\n")
        
        no_data = [r for r in results if not r['any_available']]
        if no_data:
            f.write(f"\n⚠️  Event tanpa stasiun ({len(no_data)} event):\n")
            for r in no_data[:20]:
                f.write(f"  {r['event_id']} (M{r['mag']:.1f}) at {r['time']}\n")
            if len(no_data) > 20:
                f.write(f"  ... dan {len(no_data)-20} lainnya.\n")
    
    logger.info(f"✅  Statistik disimpan ke: {output_stats}")
    
    logger.info("\n" + "="*70)
    logger.info("📊  RINGKASAN PRE-SCAN")
    logger.info("="*70)
    logger.info(f"Total event              : {stats['total']}")
    logger.info(f"IRIS saja               : {stats['iris_available']}")
    logger.info(f"GEOFON saja             : {stats['geofon_available']}")
    logger.info(f"Keduanya                : {stats['both_available']}")
    logger.info(f"Tidak ada               : {stats['none_available']}")
    logger.info(f"Minimal satu tersedia   : {stats['total'] - stats['none_available']}")
    logger.info("="*70)

# =============================================
# 7. MAIN
# =============================================

def main():
    logger.info("="*70)
    logger.info("🚀  PRE-SCAN KATALOG BMKG vs IRIS & GEOFON")
    logger.info("="*70)
    logger.info(f"📂  Katalog: {CATALOG_PATH}")
    logger.info(f"🔍  Filter : M≥{MIN_MAGNITUDE}, Tahun {YEAR_START}-{YEAR_END}")
    logger.info(f"📏  Radius : {MAX_RADIUS_DEG}°")
    logger.info(f"📡  Channel: {CHANNELS}")
    logger.info("="*70)
    
    events = read_bmkg_catalog(CATALOG_PATH)
    if not events:
        logger.error("❌  Tidak ada event.")
        sys.exit(1)
    
    results, stats = scan_events(events, MAX_RADIUS_DEG)
    save_results(results, stats, OUTPUT_CSV, OUTPUT_STATS)
    
    logger.info("✨  PRE-SCAN SELESAI!")

if __name__ == "__main__":
    main()

2026-06-20 08:16:26,914 - INFO - ======================================================================
2026-06-20 08:16:26,920 - INFO - 🚀  PRE-SCAN KATALOG BMKG vs IRIS & GEOFON
2026-06-20 08:16:26,920 - INFO - ======================================================================
2026-06-20 08:16:26,920 - INFO - 📂  Katalog: /Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog.csv
2026-06-20 08:16:26,921 - INFO - 🔍  Filter : M≥3.0, Tahun 2010-2024
2026-06-20 08:16:26,921 - INFO - 📏  Radius : 10.0°
2026-06-20 08:16:26,921 - INFO - 📡  Channel: ['BHZ', 'HHZ', 'EHZ', 'HNZ']
2026-06-20 08:16:26,922 - INFO - ======================================================================


/opt/homebrew/Caskroom/miniforge/base/envs/tf-metal/lib/python3.11/site-packages/obspy/clients/fdsn/client.py:251: ObsPyDeprecationWarning: IRIS is now EarthScope, please consider changing the FDSN client short URL to 'EARTHSCOPE'.
  warnings.warn(msg, ObsPyDeprecationWarning)


2026-06-20 08:16:27,140 - INFO - ======================================================================
2026-06-20 08:16:27,140 - INFO - 📂  MEMBACA KATALOG BMKG
2026-06-20 08:16:27,141 - INFO - ======================================================================
2026-06-20 08:16:27,141 - INFO - Kolom: ['No', 'Event ID', 'Unnamed: 2', 'Date', 'Time (UTC)', 'Latitude', 'Longitude', 'Magnitude', 'Mag Type', 'Depth (km)', 'Source', 'Source Event ID']
2026-06-20 08:16:40,440 - INFO - ✅  112063 event lolos filter (M≥3.0, 2010-2024)
2026-06-20 08:16:40,447 - INFO - 🔍  SCANNING ketersediaan stasiun...


Scanning:   0%|          | 478/112063 [52:02<202:28:25,  6.53s/event] 


KeyboardInterrupt: 

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
EDA (Exploratory Data Analysis) untuk Katalog BMKG
Tujuan: Memahami struktur, distribusi, dan potensi masalah data
sebelum digunakan untuk download waveform atau benchmarking.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from obspy import UTCDateTime
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. KONFIGURASI
# =============================================
CATALOG_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog.csv"
OUTPUT_DIR = "/Volumes/Extreme SSD/unduhan_juni_bmkg_usgs/hasilscan/eda_output'"
Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

# =============================================
# 2. BACA KATALOG
# =============================================
print("="*70)
print("📂  MEMBACA KATALOG BMKG")
print("="*70)

df = pd.read_csv(CATALOG_PATH)
df.columns = df.columns.str.strip()

print(f"✅  Jumlah baris: {len(df)}")
print(f"✅  Kolom: {df.columns.tolist()}")
print("\n📋  Contoh 5 data pertama:")
print(df.head())

# =============================================
# 3. PREPROCESS & PARSE WAKTU
# =============================================
print("\n" + "="*70)
print("🕒  PARSING WAKTU")
print("="*70)

# Konversi waktu
def parse_time(row):
    try:
        date_str = str(row['Date']).strip()
        time_str = str(row['Time (UTC)']).strip()
        if date_str == 'nan' or time_str == 'nan' or not date_str or not time_str:
            return None
        datetime_str = f"{date_str} {time_str}"
        return pd.to_datetime(datetime_str, format='%d-%b-%Y %H:%M:%S')
    except:
        return None

df['datetime'] = df.apply(parse_time, axis=1)
df = df.dropna(subset=['datetime'])
df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month

print(f"✅  Jumlah data setelah parsing: {len(df)}")
print(f"✅  Rentang waktu: {df['datetime'].min()} s/d {df['datetime'].max()}")

# =============================================
# 4. STATISTIK MAGNITUDO & KEDALAMAN
# =============================================
print("\n" + "="*70)
print("📊  STATISTIK MAGNITUDO & KEDALAMAN")
print("="*70)

# Cek kolom magnitudo dan kedalaman (nama mungkin berbeda)
mag_col = None
depth_col = None
for col in df.columns:
    if 'mag' in col.lower() or 'magnitude' in col.lower():
        mag_col = col
    if 'depth' in col.lower():
        depth_col = col

if mag_col:
    df['mag'] = pd.to_numeric(df[mag_col], errors='coerce')
    print(f"✅  Kolom magnitudo: '{mag_col}'")
    print(f"    Min: {df['mag'].min():.1f}, Max: {df['mag'].max():.1f}, Mean: {df['mag'].mean():.2f}")
    print(f"    NaN: {df['mag'].isna().sum()}")
else:
    print("❌  Kolom magnitudo tidak ditemukan!")

if depth_col:
    df['depth'] = pd.to_numeric(df[depth_col], errors='coerce')
    print(f"✅  Kolom kedalaman: '{depth_col}'")
    print(f"    Min: {df['depth'].min():.1f}, Max: {df['depth'].max():.1f}, Mean: {df['depth'].mean():.2f} km")
    print(f"    NaN: {df['depth'].isna().sum()}")
else:
    print("❌  Kolom kedalaman tidak ditemukan!")

# =============================================
# 5. DISTRIBUSI PER TAHUN
# =============================================
print("\n" + "="*70)
print("📈  DISTRIBUSI EVENT PER TAHUN")
print("="*70)

yearly_counts = df['year'].value_counts().sort_index()
print(yearly_counts)

# Plot
plt.figure(figsize=(12,4))
yearly_counts.plot(kind='bar', color='steelblue')
plt.title('Jumlah Gempa per Tahun (BMKG)')
plt.xlabel('Tahun')
plt.ylabel('Jumlah Event')
plt.grid(True, alpha=0.3)
plt.savefig(f"{OUTPUT_DIR}/yearly_distribution.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"📊  Grafik tersimpan di: {OUTPUT_DIR}/yearly_distribution.png")

# =============================================
# 6. DISTRIBUSI MAGNITUDO
# =============================================
if 'mag' in df.columns:
    plt.figure(figsize=(12,4))
    df['mag'].hist(bins=30, color='coral', edgecolor='black')
    plt.title('Distribusi Magnitudo')
    plt.xlabel('Magnitudo')
    plt.ylabel('Frekuensi')
    plt.grid(True, alpha=0.3)
    plt.savefig(f"{OUTPUT_DIR}/magnitude_distribution.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"📊  Grafik tersimpan di: {OUTPUT_DIR}/magnitude_distribution.png")

# =============================================
# 7. PETA SEBARAN (MAP)
# =============================================
print("\n" + "="*70)
print("🗺️  MEMBUAT PETA SEBARAN EVENT")
print("="*70)

if 'mag' in df.columns and 'depth' in df.columns:
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Scatter plot berdasarkan magnitudo
    scatter = ax.scatter(
        df['Longitude'], df['Latitude'],
        c=df['mag'], s=10, cmap='viridis', alpha=0.6,
        vmin=df['mag'].min(), vmax=df['mag'].max()
    )
    
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('Sebaran Gempa BMKG (warna = magnitudo)')
    ax.grid(True, alpha=0.3)
    
    # Batas wilayah Indonesia
    ax.set_xlim(94, 142)
    ax.set_ylim(-12, 8)
    
    cbar = plt.colorbar(scatter)
    cbar.set_label('Magnitudo')
    
    plt.savefig(f"{OUTPUT_DIR}/spatial_distribution.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"🗺️  Peta tersimpan di: {OUTPUT_DIR}/spatial_distribution.png")

# =============================================
# 8. CEK DUPLIKAT
# =============================================
print("\n" + "="*70)
print("🔍  CEK DUPLIKAT")
print("="*70)

# Cek duplikat berdasarkan waktu + koordinat
df['time_round'] = df['datetime'].dt.round('1min')  # toleransi 1 menit
duplicates = df.duplicated(subset=['time_round', 'Latitude', 'Longitude'], keep=False)
dup_count = duplicates.sum()
print(f"✅  Jumlah duplikat (toleransi 1 menit, lokasi sama): {dup_count}")
if dup_count > 0:
    print("⚠️  Ada duplikat! Ini bisa menyebabkan event yang sama diunduh berulang.")
    print("   Contoh duplikat:")
    print(df[duplicates][['datetime', 'Latitude', 'Longitude', 'mag']].head())

# =============================================
# 9. CEK EVENT DI LUAR WILAYAH INDONESIA
# =============================================
print("\n" + "="*70)
print("🧭  CEK EVENT DI LUAR BATAS INDONESIA")
print("="*70)

outside = df[(df['Latitude'] < -11) | (df['Latitude'] > 7.5) | 
             (df['Longitude'] < 94) | (df['Longitude'] > 141)]
print(f"✅  Event di luar bounding box Indonesia (94-141 BT, -11-7.5 LS): {len(outside)}")
if len(outside) > 0:
    print("   Contoh:")
    print(outside[['datetime', 'Latitude', 'Longitude', 'mag']].head())

# =============================================
# 10. REKOMENDASI FILTER
# =============================================
print("\n" + "="*70)
print("💡  REKOMENDASI FILTER UNTUK DOWNLOAD")
print("="*70)

# Filter yang disarankan untuk MCU-Quake
if 'mag' in df.columns and 'depth' in df.columns:
    # Magnitudo > 4.0 (agar sinyal cukup kuat)
    # Kedalaman < 100 km (gempa dangkal lebih mudah direkam)
    # Tahun 2010-2024 (data lebih lengkap)
    filtered = df[
        (df['mag'] >= 4.0) & 
        (df['depth'] <= 100) & 
        (df['year'] >= 2010) & 
        (df['year'] <= 2024)
    ]
    print(f"✅  Filter rekomendasi (M≥4.0, depth≤100km, 2010-2024):")
    print(f"    Total event: {len(filtered)}")
    print(f"    Rentang mag: {filtered['mag'].min():.1f} - {filtered['mag'].max():.1f}")
    print(f"    Rentang depth: {filtered['depth'].min():.1f} - {filtered['depth'].max():.1f} km")
    
    # Simpan daftar event yang difilter ke CSV
    filtered_path = f"{OUTPUT_DIR}/filtered_events_m4_depth100.csv"
    filtered[['datetime', 'Latitude', 'Longitude', 'depth', 'mag']].to_csv(filtered_path, index=False)
    print(f"💾  Daftar event filter disimpan di: {filtered_path}")

print("\n" + "="*70)
print("✨  EDA SELESAI! Cek folder 'catalog_eda_results' untuk grafik.")
print("="*70)

📂  MEMBACA KATALOG BMKG
✅  Jumlah baris: 217807
✅  Kolom: ['No', 'Event ID', 'Unnamed: 2', 'Date', 'Time (UTC)', 'Latitude', 'Longitude', 'Magnitude', 'Mag Type', 'Depth (km)', 'Source', 'Source Event ID']

📋  Contoh 5 data pertama:
   No                 Event ID  Unnamed: 2         Date Time (UTC)  Latitude  \
0   1  BMKG-19980109094543-001         NaN  09-Jan-1998   09:45:43     -5.39   
1   2  BMKG-19980112080550-001         NaN  12-Jan-1998   08:05:50     -3.04   
2   3  BMKG-19980118120317-001         NaN  18-Jan-1998   12:03:17     -2.65   
3   4  BMKG-19980129150524-001         NaN  29-Jan-1998   15:05:24      6.76   
4   5  BMKG-19980205061524-001         NaN  05-Feb-1998   06:15:24      6.88   

   Longitude  Magnitude Mag Type  Depth (km) Source Source Event ID  
0     126.17        4.1        -         390  RC_09               -  
1     128.36        4.4        -         100  RC_09               -  
2     132.13        4.8        -          32  RC_09               -  
3     

In [13]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
EDA KATALOG BMKG - VERSI FINAL
Menggunakan nama kolom yang sudah dikonfirmasi.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. KONFIGURASI
# =============================================
CATALOG_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog.csv"
OUTPUT_DIR = "/Volumes/Extreme SSD/unduhan_juni_bmkg_usgs/hasilscan/eda_final_output"
Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

# =============================================
# 2. BACA KATALOG
# =============================================
print("="*70)
print("📂  MEMBACA KATALOG BMKG")
print("="*70)

df = pd.read_csv(CATALOG_PATH)
df.columns = df.columns.str.strip()

# Kolom yang sudah dikonfirmasi
COL_DATE = 'Date'
COL_TIME = 'Time (UTC)'
COL_LAT = 'Latitude'
COL_LON = 'Longitude'
COL_MAG = 'Magnitude'
COL_DEPTH = 'Depth (km)'
COL_EVENT_ID = 'Event ID'

print(f"✅  Jumlah baris: {len(df)}")
print(f"✅  Kolom: {df.columns.tolist()}")

# =============================================
# 3. PARSING WAKTU (FIX)
# =============================================
print("\n" + "="*70)
print("🕒  PARSING WAKTU")
print("="*70)

def parse_time(row):
    try:
        date_str = str(row[COL_DATE]).strip()
        time_str = str(row[COL_TIME]).strip()
        if date_str == 'nan' or time_str == 'nan' or not date_str or not time_str:
            return None
        # Format: 09-Jan-1998 09:45:43
        datetime_str = f"{date_str} {time_str}"
        return pd.to_datetime(datetime_str, format='%d-%b-%Y %H:%M:%S')
    except Exception as e:
        return None

df['datetime'] = df.apply(parse_time, axis=1)
df = df.dropna(subset=['datetime'])
df['year'] = df['datetime'].dt.year

print(f"✅  Jumlah data setelah parsing: {len(df)}")
print(f"✅  Rentang waktu: {df['datetime'].min()} s/d {df['datetime'].max()}")

# =============================================
# 4. STATISTIK MAGNITUDO & KEDALAMAN
# =============================================
print("\n" + "="*70)
print("📊  STATISTIK MAGNITUDO & KEDALAMAN")
print("="*70)

# Pastikan kolom magnitudo dan kedalaman terbaca
df['mag'] = pd.to_numeric(df[COL_MAG], errors='coerce')
df['depth'] = pd.to_numeric(df[COL_DEPTH], errors='coerce')

print(f"✅  Magnitudo:")
print(f"    Min: {df['mag'].min():.1f}, Max: {df['mag'].max():.1f}, Mean: {df['mag'].mean():.2f}")
print(f"    NaN: {df['mag'].isna().sum()}")
print(f"✅  Kedalaman:")
print(f"    Min: {df['depth'].min():.1f}, Max: {df['depth'].max():.1f}, Mean: {df['depth'].mean():.2f} km")
print(f"    NaN: {df['depth'].isna().sum()}")

# =============================================
# 5. FILTER DENGAN BERBAGAI THRESHOLD
# =============================================
print("\n" + "="*70)
print("🎯  HASIL FILTER DENGAN BERBAGAI THRESHOLD")
print("="*70)

# 5a. Filter M≥4.0, Depth≤100 km, 2010-2024
filter1 = df[(df['mag'] >= 4.0) & (df['depth'] <= 100) & (df['year'] >= 2010) & (df['year'] <= 2024)]
print(f"1. M≥4.0, depth≤100km, 2010-2024: {len(filter1)} event")

# 5b. Filter M≥4.5, Depth≤150 km, 2010-2024
filter2 = df[(df['mag'] >= 4.5) & (df['depth'] <= 150) & (df['year'] >= 2010) & (df['year'] <= 2024)]
print(f"2. M≥4.5, depth≤150km, 2010-2024: {len(filter2)} event")

# 5c. Filter M≥5.0, Depth≤200 km, 2010-2024
filter3 = df[(df['mag'] >= 5.0) & (df['depth'] <= 200) & (df['year'] >= 2010) & (df['year'] <= 2024)]
print(f"3. M≥5.0, depth≤200km, 2010-2024: {len(filter3)} event")

# 5d. Tanpa filter magnitudo (hanya tahun & depth)
filter4 = df[(df['depth'] <= 100) & (df['year'] >= 2010) & (df['year'] <= 2024)]
print(f"4. Semua magnitudo, depth≤100km, 2010-2024: {len(filter4)} event")

# 5e. Tanpa filter depth (hanya magnitudo & tahun)
filter5 = df[(df['mag'] >= 4.0) & (df['year'] >= 2010) & (df['year'] <= 2024)]
print(f"5. M≥4.0, semua depth, 2010-2024: {len(filter5)} event")

# =============================================
# 6. SIMPAN HASIL FILTER TERBAIK
# =============================================
# Pilih filter yang paling banyak eventnya (misal #2 atau #5)
# Sesuaikan dengan kebutuhan Anda (untuk MCU-Quake, lebih banyak data lebih baik)

chosen_filter = filter2  # M≥4.5, depth≤150km, 2010-2024
if len(chosen_filter) == 0:
    print("\n⚠️  Tidak ada event untuk filter yang dipilih. Gunakan filter #5 atau #4.")
    chosen_filter = filter5  # fallback

output_path = f"{OUTPUT_DIR}/filtered_events_selected.csv"
chosen_filter[['datetime', 'Latitude', 'Longitude', 'depth', 'mag']].to_csv(output_path, index=False)
print(f"\n💾  {len(chosen_filter)} event tersimpan di: {output_path}")

# =============================================
# 7. VISUALISASI PER TAHUN UNTUK M≥4.5
# =============================================
if len(filter2) > 0:
    plt.figure(figsize=(12,4))
    filter2['year'].value_counts().sort_index().plot(kind='bar', color='teal')
    plt.title('Jumlah Gempa M≥4.5 per Tahun (2010-2024)')
    plt.xlabel('Tahun')
    plt.ylabel('Jumlah Event')
    plt.grid(True, alpha=0.3)
    plt.savefig(f"{OUTPUT_DIR}/yearly_M4.5.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"📊  Grafik tersimpan di: {OUTPUT_DIR}/yearly_M4.5.png")

print("\n" + "="*70)
print("✨  EDA SELESAI!")
print("="*70)

📂  MEMBACA KATALOG BMKG
✅  Jumlah baris: 217807
✅  Kolom: ['No', 'Event ID', 'Unnamed: 2', 'Date', 'Time (UTC)', 'Latitude', 'Longitude', 'Magnitude', 'Mag Type', 'Depth (km)', 'Source', 'Source Event ID']

🕒  PARSING WAKTU
✅  Jumlah data setelah parsing: 217807
✅  Rentang waktu: 1998-01-09 09:45:43 s/d 2024-12-31 23:58:59

📊  STATISTIK MAGNITUDO & KEDALAMAN
✅  Magnitudo:
    Min: 0.2, Max: 7.9, Mean: 3.15
    NaN: 0
✅  Kedalaman:
    Min: 1.0, Max: 798.0, Mean: 43.24 km
    NaN: 0

🎯  HASIL FILTER DENGAN BERBAGAI THRESHOLD
1. M≥4.0, depth≤100km, 2010-2024: 26940 event
2. M≥4.5, depth≤150km, 2010-2024: 14248 event
3. M≥5.0, depth≤200km, 2010-2024: 3816 event
4. Semua magnitudo, depth≤100km, 2010-2024: 184805 event
5. M≥4.0, semua depth, 2010-2024: 37449 event

💾  14248 event tersimpan di: /Volumes/Extreme SSD/unduhan_juni_bmkg_usgs/hasilscan/eda_final_output/filtered_events_selected.csv
📊  Grafik tersimpan di: /Volumes/Extreme SSD/unduhan_juni_bmkg_usgs/hasilscan/eda_final_output/yearl

In [18]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
DOWNLOAD WAVEFORM PRIORITAS GEOFON (GFZ) UNTUK DATA INDONESIA
Mengunduh data 3 komponen dari jaringan GE, IA, dan IRIS sebagai cadangan.
"""

import os
import sys
import time
import pandas as pd
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.geodetics import gps2dist_azimuth
from tqdm import tqdm
import logging

# =============================================
# 1. KONFIGURASI
# =============================================

CATALOG_CSV = "/Volumes/Extreme SSD/unduhan_juni_bmkg_usgs/hasilscan/eda_final_output/filtered_events_selected.csv"
OUTPUT_DIR = "/Volumes/Extreme SSD/unduhan_waveform_geofon"

TIME_BEFORE = 30
TIME_AFTER = 120
MAX_RADIUS_DEG = 12.0          # radius awal
FALLBACK_RADIUS_DEG = 25.0     # jika gagal, coba radius lebih besar

# Prioritas channel: GEOFON biasanya pakai BH?, HH?, tapi ada juga EH?
CHANNELS = ['BHZ', 'BHN', 'BHE', 'HHZ', 'HHN', 'HHE', 'EHZ', 'EHN', 'EHE']
LOCATION = "*"

# Jaringan prioritas (GEOFON dan BMKG-GEOFON)
PRIORITY_NETWORKS = ['GE', 'IA', 'IU', 'II', 'GT']

MAX_EVENTS = 10  # testing, set None untuk semua

# =============================================
# 2. SETUP LOGGING
# =============================================

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# =============================================
# 3. FUNGSI UTAMA
# =============================================

def find_station_geofon(lat, lon, origin_time, radius_deg, channels):
    """
    Cari stasiun dari GEOFON (jaringan GE, IA) terdekat.
    Return (network, station, distance_deg) atau (None, None, None)
    """
    client = Client("GEOFON")
    try:
        inventory = client.get_stations(
            latitude=lat,
            longitude=lon,
            maxradius=radius_deg,
            level="channel",
            channel=",".join(channels),
            location=LOCATION,
            starttime=origin_time - 60,
            endtime=origin_time + 60,
            limit=50,
            network=",".join(PRIORITY_NETWORKS)  # filter jaringan
        )
        if not inventory:
            return None, None, None

        best_dist = float('inf')
        best_net = None
        best_sta = None
        for network in inventory:
            for station in network:
                if station.latitude is None or station.longitude is None:
                    continue
                dist_m, _, _ = gps2dist_azimuth(lat, lon, station.latitude, station.longitude)
                dist_deg = dist_m / 111000.0
                if dist_deg < best_dist:
                    best_dist = dist_deg
                    best_net = network.code
                    best_sta = station.code
        if best_net and best_sta:
            return best_net, best_sta, best_dist
        return None, None, None
    except Exception as e:
        logger.debug(f"Error finding GEOFON station: {e}")
        return None, None, None

def find_station_iris(lat, lon, origin_time, radius_deg, channels):
    """
    Cari stasiun dari IRIS (jaringan IU, II) sebagai fallback.
    """
    client = Client("IRIS")
    try:
        inventory = client.get_stations(
            latitude=lat,
            longitude=lon,
            maxradius=radius_deg,
            level="channel",
            channel=",".join(channels),
            location=LOCATION,
            starttime=origin_time - 60,
            endtime=origin_time + 60,
            limit=50
        )
        if not inventory:
            return None, None, None

        best_dist = float('inf')
        best_net = None
        best_sta = None
        for network in inventory:
            # Filter agar hanya jaringan yang relevan (IU, II, GT)
            if network.code not in ['IU', 'II', 'GT']:
                continue
            for station in network:
                if station.latitude is None or station.longitude is None:
                    continue
                dist_m, _, _ = gps2dist_azimuth(lat, lon, station.latitude, station.longitude)
                dist_deg = dist_m / 111000.0
                if dist_deg < best_dist:
                    best_dist = dist_deg
                    best_net = network.code
                    best_sta = station.code
        if best_net and best_sta:
            return best_net, best_sta, best_dist
        return None, None, None
    except Exception as e:
        logger.debug(f"Error finding IRIS station: {e}")
        return None, None, None

def download_event(event, output_dir):
    """
    Unduh waveform untuk satu event dengan prioritas GEOFON.
    """
    lat = event['lat']
    lon = event['lon']
    origin = event['time']
    event_id = event['id']

    # --- 1. Coba GEOFON dulu ---
    net, sta, dist = find_station_geofon(lat, lon, origin, MAX_RADIUS_DEG, CHANNELS)

    # Jika GEOFON gagal, coba radius lebih besar
    if not net:
        logger.info(f"Event {event_id}: GEOFON radius {MAX_RADIUS_DEG}° gagal, coba {FALLBACK_RADIUS_DEG}°...")
        net, sta, dist = find_station_geofon(lat, lon, origin, FALLBACK_RADIUS_DEG, CHANNELS)

    # --- 2. Jika GEOFON masih gagal, coba IRIS ---
    if not net:
        logger.info(f"Event {event_id}: GEOFON tidak ditemukan, coba IRIS...")
        net, sta, dist = find_station_iris(lat, lon, origin, MAX_RADIUS_DEG, CHANNELS)
        if not net:
            # IRIS radius besar
            net, sta, dist = find_station_iris(lat, lon, origin, FALLBACK_RADIUS_DEG, CHANNELS)

    if not net:
        logger.warning(f"Event {event_id}: Tidak ada stasiun dalam radius {FALLBACK_RADIUS_DEG}° (GEOFON/IRIS)")
        return False

    starttime = origin - TIME_BEFORE
    endtime = origin + TIME_AFTER

    # --- 3. Unduh data dari client yang sesuai ---
    # Tentukan client berdasarkan jaringan yang ditemukan
    if net in ['GE', 'IA']:
        client = Client("GEOFON")
    else:
        client = Client("IRIS")

    stream = None
    for ch in CHANNELS:
        try:
            st = client.get_waveforms(
                network=net,
                station=sta,
                location=LOCATION,
                channel=ch,
                starttime=starttime,
                endtime=endtime
            )
            if st and len(st) > 0:
                if stream is None:
                    stream = st
                else:
                    stream += st
        except Exception:
            continue

    if stream is None or len(stream) == 0:
        logger.warning(f"Event {event_id}: Tidak ada data untuk {net}.{sta}")
        return False

    os.makedirs(output_dir, exist_ok=True)
    filename = f"{net}_{sta}_{event_id}.mseed"
    filepath = os.path.join(output_dir, filename)

    try:
        stream.write(filepath, format="MSEED")
        logger.info(f"Event {event_id}: Berhasil disimpan ke {filename} (trace: {len(stream)}, net: {net})")
        return True
    except Exception as e:
        logger.error(f"Event {event_id}: Gagal menyimpan: {e}")
        return False

def locate_catalog_file(catalog_path):
    """Cari file katalog."""
    if os.path.exists(catalog_path):
        return catalog_path
    try:
        script_dir = os.path.dirname(os.path.abspath(__file__))
        alt_path = os.path.join(script_dir, "filtered_events_selected.csv")
        if os.path.exists(alt_path):
            return alt_path
    except NameError:
        pass
    cwd_path = os.path.join(os.getcwd(), "filtered_events_selected.csv")
    if os.path.exists(cwd_path):
        return cwd_path
    user_input = input("Masukkan path file CSV (Enter untuk keluar): ").strip()
    if user_input and os.path.exists(user_input):
        return user_input
    sys.exit(1)

def main():
    logger.info("="*60)
    logger.info("🚀 DOWNLOAD WAVEFORM PRIORITAS GEOFON")
    logger.info("="*60)

    global CATALOG_CSV
    CATALOG_CSV = locate_catalog_file(CATALOG_CSV)
    logger.info(f"📂 Katalog: {CATALOG_CSV}")

    df = pd.read_csv(CATALOG_CSV)
    logger.info(f"✅ Total event: {len(df)}")
    logger.info(f"📋 Kolom: {df.columns.tolist()}")

    # Deteksi kolom
    time_col = next((c for c in df.columns if 'time' in c.lower() or 'datetime' in c.lower()), None)
    lat_col = next((c for c in df.columns if 'lat' in c.lower()), None)
    lon_col = next((c for c in df.columns if 'lon' in c.lower()), None)
    mag_col = next((c for c in df.columns if 'mag' in c.lower()), 'mag')
    depth_col = next((c for c in df.columns if 'depth' in c.lower()), 'depth')

    if not all([time_col, lat_col, lon_col]):
        logger.error("❌ Kolom waktu/lat/lon tidak ditemukan.")
        sys.exit(1)

    df['datetime'] = pd.to_datetime(df[time_col], utc=True)
    df = df.dropna(subset=['datetime'])

    if MAX_EVENTS and len(df) > MAX_EVENTS:
        df = df.head(MAX_EVENTS)
        logger.info(f"⚠️ Hanya {MAX_EVENTS} event pertama.")

    success = 0
    failed = 0
    total = len(df)

    for idx, row in tqdm(df.iterrows(), total=total, desc="Mengunduh"):
        origin = UTCDateTime(row['datetime'])
        lat = row[lat_col]
        lon = row[lon_col]
        event_id = origin.strftime("%Y%m%d_%H%M%S")

        event = {
            'id': event_id,
            'time': origin,
            'lat': lat,
            'lon': lon,
            'mag': row.get(mag_col, 0),
            'depth': row.get(depth_col, 0)
        }

        if download_event(event, OUTPUT_DIR):
            success += 1
        else:
            failed += 1

        time.sleep(0.5)

    logger.info("="*60)
    logger.info(f"✨ SELESAI! Berhasil: {success}, Gagal: {failed}, Total: {total}")
    logger.info(f"📁 Data di: {OUTPUT_DIR}")
    logger.info("="*60)

if __name__ == "__main__":
    main()

2026-06-20 09:37:36,701 - INFO - ============================================================
2026-06-20 09:37:36,701 - INFO - 🚀 DOWNLOAD WAVEFORM PRIORITAS GEOFON
2026-06-20 09:37:36,702 - INFO - ============================================================
2026-06-20 09:37:36,702 - INFO - 📂 Katalog: /Volumes/Extreme SSD/unduhan_juni_bmkg_usgs/hasilscan/eda_final_output/filtered_events_selected.csv
2026-06-20 09:37:36,716 - INFO - ✅ Total event: 14248
2026-06-20 09:37:36,717 - INFO - 📋 Kolom: ['datetime', 'Latitude', 'Longitude', 'depth', 'mag']
2026-06-20 09:37:36,721 - INFO - ⚠️ Hanya 10 event pertama.
Mengunduh:   0%|          | 0/10 [00:00<?, ?it/s]2026-06-20 09:37:38,490 - INFO - Event 20100101_044258: GEOFON radius 12.0° gagal, coba 25.0°...
2026-06-20 09:37:38,491 - INFO - Event 20100101_044258: GEOFON tidak ditemukan, coba IRIS...
2026-06-20 09:37:38,492 - WARNING - Event 20100101_044258: Tidak ada stasiun dalam radius 25.0° (GEOFON/IRIS)
Mengunduh:  10%|█         | 1/10 [00:02

In [21]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
DOWNLOAD WAVEFORM PRIORITAS GEOFON
Mengunduh data 3 komponen (BHZ/BHN/BHE) dari stasiun GEOFON terdekat.
"""

import os
import sys
import time
import pandas as pd
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.geodetics import gps2dist_azimuth
from tqdm import tqdm
import logging

# =============================================
# 1. KONFIGURASI
# =============================================

CATALOG_CSV = "/Volumes/Extreme SSD/unduhan_juni_bmkg_usgs/hasilscan/eda_final_output/filtered_events_selected.csv"
OUTPUT_DIR = "/Volumes/Extreme SSD/unduhan_waveform_geofon"

TIME_BEFORE = 30        # detik sebelum origin
TIME_AFTER = 120        # detik setelah origin
MAX_RADIUS_DEG = 15.0   # radius pencarian stasiun
CHANNELS = ['BHZ', 'BHN', 'BHE']  # broadband channels
LOCATION = "*"          # wildcard

MAX_EVENTS = 10         # untuk testing; set None untuk semua

# =============================================
# 2. SETUP LOGGING
# =============================================

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# =============================================
# 3. FUNGSI UTAMA
# =============================================

def find_station_geofon(lat, lon, origin_time, radius_deg, channels):
    """
    Cari stasiun GEOFON terdekat dengan channel yang diinginkan.
    Return (network, station, distance_deg) atau (None, None, None)
    """
    client = Client("GEOFON")
    try:
        # Tanpa filter waktu agar lebih fleksibel (cari stasiun yang aktif)
        inventory = client.get_stations(
            latitude=lat,
            longitude=lon,
            maxradius=radius_deg,
            level="channel",
            channel=",".join(channels)
        )
        if not inventory:
            return None, None, None

        best_dist = float('inf')
        best_net = None
        best_sta = None
        for network in inventory:
            for station in network:
                if station.latitude is None or station.longitude is None:
                    continue
                dist_m, _, _ = gps2dist_azimuth(lat, lon, station.latitude, station.longitude)
                dist_deg = dist_m / 111000.0
                if dist_deg < best_dist:
                    best_dist = dist_deg
                    best_net = network.code
                    best_sta = station.code
        if best_net and best_sta:
            return best_net, best_sta, best_dist
        return None, None, None
    except Exception as e:
        logger.debug(f"Error finding GEOFON station: {e}")
        return None, None, None

def download_event_geofon(event, output_dir):
    """
    Unduh waveform untuk satu event dari GEOFON (prioritas).
    """
    lat = event['lat']
    lon = event['lon']
    origin = event['time']
    event_id = event['id']

    # Cari stasiun GEOFON terdekat
    net, sta, dist = find_station_geofon(lat, lon, origin, MAX_RADIUS_DEG, CHANNELS)
    if not net:
        logger.warning(f"Event {event_id}: Tidak ada stasiun GEOFON dalam radius {MAX_RADIUS_DEG}°")
        return False

    # Kalau jarak cukup dekat, unduh
    logger.info(f"Event {event_id}: Stasiun {net}.{sta} (jarak {dist:.2f}°)")

    starttime = origin - TIME_BEFORE
    endtime = origin + TIME_AFTER

    client = Client("GEOFON")
    stream = None
    for ch in CHANNELS:
        try:
            st = client.get_waveforms(
                network=net,
                station=sta,
                location=LOCATION,
                channel=ch,
                starttime=starttime,
                endtime=endtime
            )
            if st and len(st) > 0:
                if stream is None:
                    stream = st
                else:
                    stream += st
        except Exception as e:
            # Gagal untuk channel ini, lanjut
            continue

    if stream is None or len(stream) == 0:
        logger.warning(f"Event {event_id}: Tidak ada data untuk {net}.{sta}")
        return False

    os.makedirs(output_dir, exist_ok=True)
    filename = f"{net}_{sta}_{event_id}.mseed"
    filepath = os.path.join(output_dir, filename)

    try:
        stream.write(filepath, format="MSEED")
        logger.info(f"Event {event_id}: Berhasil disimpan ke {filename} (trace: {len(stream)})")
        return True
    except Exception as e:
        logger.error(f"Event {event_id}: Gagal menyimpan: {e}")
        return False

def locate_catalog_file(catalog_path):
    if os.path.exists(catalog_path):
        return catalog_path
    # Cari di direktori kerja
    cwd_path = os.path.join(os.getcwd(), "filtered_events_selected.csv")
    if os.path.exists(cwd_path):
        return cwd_path
    user_input = input("Masukkan path lengkap ke file filtered_events_selected.csv: ").strip()
    if user_input and os.path.exists(user_input):
        return user_input
    else:
        logger.error("File tidak ditemukan. Keluar.")
        sys.exit(1)

def main():
    logger.info("="*60)
    logger.info("🚀 DOWNLOAD WAVEFORM PRIORITAS GEOFON")
    logger.info("="*60)

    global CATALOG_CSV
    CATALOG_CSV = locate_catalog_file(CATALOG_CSV)
    logger.info(f"📂 Katalog: {CATALOG_CSV}")

    df = pd.read_csv(CATALOG_CSV)
    logger.info(f"✅ Total event: {len(df)}")
    logger.info(f"📋 Kolom: {df.columns.tolist()}")

    # Deteksi kolom
    time_col = 'datetime'
    lat_col = 'Latitude'
    lon_col = 'Longitude'
    mag_col = 'mag'
    depth_col = 'depth'

    df['datetime'] = pd.to_datetime(df[time_col], utc=True)
    df = df.dropna(subset=['datetime'])
    df['mag'] = pd.to_numeric(df[mag_col], errors='coerce').fillna(0)
    df['depth'] = pd.to_numeric(df[depth_col], errors='coerce').fillna(0)

    if MAX_EVENTS and len(df) > MAX_EVENTS:
        df = df.head(MAX_EVENTS)
        logger.info(f"⚠️ Hanya {MAX_EVENTS} event pertama.")

    success = 0
    failed = 0
    total = len(df)

    for idx, row in tqdm(df.iterrows(), total=total, desc="Mengunduh"):
        origin = UTCDateTime(row['datetime'])
        lat = row[lat_col]
        lon = row[lon_col]
        mag = row['mag']
        depth = row['depth']
        event_id = origin.strftime("%Y%m%d_%H%M%S")

        event = {
            'id': event_id,
            'time': origin,
            'lat': lat,
            'lon': lon,
            'mag': mag,
            'depth': depth
        }

        if download_event_geofon(event, OUTPUT_DIR):
            success += 1
        else:
            failed += 1

        time.sleep(0.5)

    logger.info("="*60)
    logger.info(f"✨ SELESAI! Berhasil: {success}, Gagal: {failed}, Total: {total}")
    logger.info(f"📁 Data di: {OUTPUT_DIR}")
    logger.info("="*60)

if __name__ == "__main__":
    main()

2026-06-20 09:40:21,246 - INFO - ============================================================
2026-06-20 09:40:21,247 - INFO - 🚀 DOWNLOAD WAVEFORM PRIORITAS GEOFON
2026-06-20 09:40:21,247 - INFO - ============================================================
2026-06-20 09:40:21,247 - INFO - 📂 Katalog: /Volumes/Extreme SSD/unduhan_juni_bmkg_usgs/hasilscan/eda_final_output/filtered_events_selected.csv
2026-06-20 09:40:21,262 - INFO - ✅ Total event: 14248
2026-06-20 09:40:21,262 - INFO - 📋 Kolom: ['datetime', 'Latitude', 'Longitude', 'depth', 'mag']
2026-06-20 09:40:21,267 - INFO - ⚠️ Hanya 10 event pertama.
Mengunduh:   0%|          | 0/10 [00:00<?, ?it/s]2026-06-20 09:40:35,101 - INFO - Event 20100101_044258: Stasiun GE.TNTI (jarak 4.04°)
2026-06-20 09:40:37,283 - INFO - Event 20100101_044258: Berhasil disimpan ke GE_TNTI_20100101_044258.mseed (trace: 3)
Mengunduh:  10%|█         | 1/10 [00:16<02:28, 16.52s/it]2026-06-20 09:40:38,679 - INFO - Event 20100101_135238: Stasiun GE.TNTI (jarak

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
DOWNLOAD WAVEFORM PRIORITAS GEOFON - PARALLEL VERSION
Mengunduh 3 komponen (BH?/HH?) untuk setiap event di katalog.
Dengan caching stasiun, parallel download, dan resume otomatis.
"""

import os
import sys
import time
import pandas as pd
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.geodetics import gps2dist_azimuth
from tqdm import tqdm
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import lru_cache
import hashlib

# =============================================
# KONFIGURASI
# =============================================

CATALOG_CSV = "/Volumes/Extreme SSD/unduhan_juni_bmkg_usgs/hasilscan/eda_final_output/filtered_events_selected.csv"
OUTPUT_DIR = "/Volumes/Extreme SSD/unduhan_waveform_geofon"

TIME_BEFORE = 30          # detik sebelum origin
TIME_AFTER = 120          # detik setelah origin
MAX_RADIUS_DEG = 12.0     # radius pencarian stasiun awal
FALLBACK_RADIUS_DEG = 25.0 # radius kedua jika gagal
CHANNELS = ['BHZ','BHN','BHE','HHZ','HHN','HHE','EHZ','EHN','EHE']
LOCATION = "*"

# --- Parallel ---
MAX_WORKERS = 6           # jumlah thread paralel (sesuaikan)
MAX_EVENTS = None         # None untuk semua, atau angka untuk testing

# --- Logging ---
LOG_FILE = "download_waveform.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# =============================================
# CACHE STASIUN (agar tidak query ulang)
# =============================================
station_cache = {}

def cache_key(client_name, lat, lon, radius_deg, year):
    """Buat key untuk cache stasiun."""
    return f"{client_name}_{round(lat,2)}_{round(lon,2)}_{radius_deg}_{year}"

def find_station(client, client_name, lat, lon, origin_time, radius_deg, channels):
    """
    Cari stasiun terdekat dengan channel yang diinginkan.
    Return (network, station, distance_deg) atau (None, None, None)
    """
    year = origin_time.year
    key = cache_key(client_name, lat, lon, radius_deg, year)
    
    if key in station_cache:
        return station_cache[key]
    
    try:
        inventory = client.get_stations(
            latitude=lat,
            longitude=lon,
            maxradius=radius_deg,
            level="channel",
            channel=",".join(channels),
            location=LOCATION,
            starttime=origin_time - 60,
            endtime=origin_time + 60
        )
        if not inventory:
            station_cache[key] = (None, None, None)
            return None, None, None

        best_dist = float('inf')
        best_net = None
        best_sta = None
        for network in inventory:
            for station in network:
                if station.latitude is None or station.longitude is None:
                    continue
                dist_m, _, _ = gps2dist_azimuth(lat, lon, station.latitude, station.longitude)
                dist_deg = dist_m / 111000.0
                if dist_deg < best_dist:
                    best_dist = dist_deg
                    best_net = network.code
                    best_sta = station.code
        if best_net and best_sta:
            result = (best_net, best_sta, best_dist)
            station_cache[key] = result
            return result
        else:
            station_cache[key] = (None, None, None)
            return None, None, None
    except Exception as e:
        logger.debug(f"Error finding station: {e}")
        station_cache[key] = (None, None, None)
        return None, None, None

# =============================================
# FUNGSI DOWNLOAD SATU EVENT
# =============================================

def download_event(event, output_dir):
    """
    Unduh waveform untuk satu event.
    Return True jika berhasil, False jika gagal.
    """
    lat = event['lat']
    lon = event['lon']
    origin = event['time']
    event_id = event['id']

    os.makedirs(output_dir, exist_ok=True)

    # Cek apakah file sudah ada (resume)
    existing = [f for f in os.listdir(output_dir) if event_id in f and f.endswith('.mseed')]
    if existing:
        logger.info(f"Event {event_id}: File sudah ada, skip.")
        return True

    # --- 1. Coba GEOFON ---
    client_geofon = Client("GEOFON")
    net, sta, dist = find_station(client_geofon, "GEOFON", lat, lon, origin, MAX_RADIUS_DEG, CHANNELS)

    if not net:
        logger.info(f"Event {event_id}: GEOFON radius {MAX_RADIUS_DEG}° gagal, coba {FALLBACK_RADIUS_DEG}°...")
        net, sta, dist = find_station(client_geofon, "GEOFON", lat, lon, origin, FALLBACK_RADIUS_DEG, CHANNELS)

    # --- 2. Jika gagal, coba IRIS ---
    client_iris = Client("IRIS")
    if not net:
        logger.info(f"Event {event_id}: GEOFON tidak ditemukan, coba IRIS...")
        net, sta, dist = find_station(client_iris, "IRIS", lat, lon, origin, MAX_RADIUS_DEG, CHANNELS)
        if not net:
            net, sta, dist = find_station(client_iris, "IRIS", lat, lon, origin, FALLBACK_RADIUS_DEG, CHANNELS)
    
    # --- 3. Jika tetap gagal ---
    if not net:
        logger.warning(f"Event {event_id}: Tidak ada stasiun dalam radius {FALLBACK_RADIUS_DEG}° (GEOFON/IRIS)")
        return False

    logger.info(f"Event {event_id}: Stasiun {net}.{sta} (jarak {dist:.2f}°)")

    starttime = origin - TIME_BEFORE
    endtime = origin + TIME_AFTER

    # Pilih client terakhir yang berhasil
    if net and client_geofon:
        client = client_geofon
    else:
        client = client_iris

    # --- Unduh data untuk semua channel ---
    stream = None
    for ch in CHANNELS:
        try:
            st = client.get_waveforms(
                network=net,
                station=sta,
                location=LOCATION,
                channel=ch,
                starttime=starttime,
                endtime=endtime
            )
            if st and len(st) > 0:
                if stream is None:
                    stream = st
                else:
                    stream += st
        except Exception as e:
            # Gagal untuk channel ini, lanjutkan
            continue

    if stream is None or len(stream) == 0:
        logger.warning(f"Event {event_id}: Tidak ada data untuk {net}.{sta}")
        return False

    # --- Simpan ---
    filename = f"{net}_{sta}_{event_id}.mseed"
    filepath = os.path.join(output_dir, filename)
    try:
        stream.write(filepath, format="MSEED")
        logger.info(f"Event {event_id}: Berhasil disimpan ke {filename} (trace: {len(stream)})")
        return True
    except Exception as e:
        logger.error(f"Event {event_id}: Gagal menyimpan: {e}")
        return False

# =============================================
# FUNGSI MEMBACA KATALOG
# =============================================

def locate_catalog_file(catalog_path):
    """Cari file katalog di beberapa lokasi."""
    if os.path.exists(catalog_path):
        return catalog_path

    # Cari di direktori script (jika ada)
    try:
        script_dir = os.path.dirname(os.path.abspath(__file__))
        alt_path = os.path.join(script_dir, "filtered_events_selected.csv")
        if os.path.exists(alt_path):
            return alt_path
    except NameError:
        pass  # di Jupyter/REPL

    # Cari di direktori kerja
    cwd_path = os.path.join(os.getcwd(), "filtered_events_selected.csv")
    if os.path.exists(cwd_path):
        return cwd_path

    # Minta input user
    user_input = input("Masukkan path lengkap ke file filtered_events_selected.csv (atau Enter untuk keluar): ").strip()
    if user_input and os.path.exists(user_input):
        return user_input
    else:
        logger.error("File tidak ditemukan. Keluar.")
        sys.exit(1)

def read_catalog(catalog_path):
    """Baca dan validasi katalog."""
    df = pd.read_csv(catalog_path)
    logger.info(f"✅ Total event dalam katalog: {len(df)}")
    logger.info(f"📋 Kolom yang tersedia: {df.columns.tolist()}")

    # Deteksi kolom waktu
    time_col = None
    for col in df.columns:
        if 'time' in col.lower() or 'datetime' in col.lower():
            time_col = col
            break
    if time_col is None:
        logger.error("❌ Kolom waktu tidak ditemukan.")
        sys.exit(1)
    logger.info(f"🕒 Kolom waktu: '{time_col}'")

    # Deteksi kolom latitude
    lat_col = None
    for col in df.columns:
        if 'lat' in col.lower():
            lat_col = col
            break
    if lat_col is None:
        logger.error("❌ Kolom latitude tidak ditemukan.")
        sys.exit(1)
    logger.info(f"🌐 Kolom latitude: '{lat_col}'")

    # Deteksi kolom longitude
    lon_col = None
    for col in df.columns:
        if 'lon' in col.lower():
            lon_col = col
            break
    if lon_col is None:
        logger.error("❌ Kolom longitude tidak ditemukan.")
        sys.exit(1)
    logger.info(f"🌐 Kolom longitude: '{lon_col}'")

    # Konversi datetime
    df['datetime'] = pd.to_datetime(df[time_col], utc=True)
    df = df.dropna(subset=['datetime'])

    return df, lat_col, lon_col

# =============================================
# MAIN
# =============================================

def main():
    logger.info("="*60)
    logger.info("🚀 DOWNLOAD WAVEFORM - PARALLEL (GEOFON/IRIS)")
    logger.info("="*60)

    # 1. Temukan file katalog
    global CATALOG_CSV
    CATALOG_CSV = locate_catalog_file(CATALOG_CSV)
    logger.info(f"📂 Katalog: {CATALOG_CSV}")

    # 2. Baca katalog
    df, lat_col, lon_col = read_catalog(CATALOG_CSV)

    if MAX_EVENTS and len(df) > MAX_EVENTS:
        df = df.head(MAX_EVENTS)
        logger.info(f"⚠️ Hanya {MAX_EVENTS} event pertama yang diproses.")
    else:
        logger.info(f"📦 Total event yang akan diproses: {len(df)}")

    # 3. Siapkan daftar event
    events = []
    for _, row in df.iterrows():
        origin = UTCDateTime(row['datetime'])
        events.append({
            'id': origin.strftime("%Y%m%d_%H%M%S"),
            'time': origin,
            'lat': row[lat_col],
            'lon': row[lon_col]
        })

    logger.info(f"⚡ Parallel workers: {MAX_WORKERS}")
    logger.info(f"📁 Output directory: {OUTPUT_DIR}")

    # 4. Proses unduhan paralel
    success = 0
    failed = 0
    total = len(events)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(download_event, ev, OUTPUT_DIR): ev for ev in events}
        with tqdm(total=len(futures), desc="Mengunduh", unit="event") as pbar:
            for future in as_completed(futures):
                try:
                    result = future.result()
                    if result:
                        success += 1
                    else:
                        failed += 1
                except Exception as e:
                    logger.error(f"Error pada event: {e}")
                    failed += 1
                pbar.update(1)

    logger.info("="*60)
    logger.info(f"✨ SELESAI! Berhasil: {success}, Gagal: {failed}, Total: {total}")
    logger.info(f"📁 Data disimpan di: {OUTPUT_DIR}")
    logger.info(f"📄 Log tersimpan di: {LOG_FILE}")
    logger.info("="*60)

if __name__ == "__main__":
    main()

2026-06-20 09:56:44,740 - INFO - ============================================================
2026-06-20 09:56:44,741 - INFO - 🚀 DOWNLOAD WAVEFORM - PARALLEL (GEOFON/IRIS)
2026-06-20 09:56:44,741 - INFO - ============================================================
2026-06-20 09:56:44,741 - INFO - 📂 Katalog: /Volumes/Extreme SSD/unduhan_juni_bmkg_usgs/hasilscan/eda_final_output/filtered_events_selected.csv
2026-06-20 09:56:44,755 - INFO - ✅ Total event dalam katalog: 14248
2026-06-20 09:56:44,756 - INFO - 📋 Kolom yang tersedia: ['datetime', 'Latitude', 'Longitude', 'depth', 'mag']
2026-06-20 09:56:44,756 - INFO - 🕒 Kolom waktu: 'datetime'
2026-06-20 09:56:44,756 - INFO - 🌐 Kolom latitude: 'Latitude'
2026-06-20 09:56:44,756 - INFO - 🌐 Kolom longitude: 'Longitude'
2026-06-20 09:56:44,761 - INFO - 📦 Total event yang akan diproses: 14248
2026-06-20 09:56:45,105 - INFO - ⚡ Parallel workers: 6
2026-06-20 09:56:45,106 - INFO - 📁 Output directory: /Volumes/Extreme SSD/unduhan_waveform_geofon
2

Mengunduh:   0%|          | 0/14248 [00:00<?, ?event/s]

2026-06-20 09:56:45,210 - INFO - Event 20100104_205909: File sudah ada, skip.
2026-06-20 09:56:45,211 - INFO - Event 20100104_044639: File sudah ada, skip.
2026-06-20 09:56:45,211 - INFO - Event 20100104_225605: File sudah ada, skip.
2026-06-20 09:56:45,221 - INFO - Event 20100104_180744: File sudah ada, skip.
2026-06-20 09:56:45,223 - INFO - Event 20100105_105421: File sudah ada, skip.
2026-06-20 09:56:45,224 - INFO - Event 20100105_170958: File sudah ada, skip.
2026-06-20 09:56:45,224 - INFO - Event 20100105_143627: File sudah ada, skip.
2026-06-20 09:56:45,225 - INFO - Event 20100105_214614: File sudah ada, skip.
2026-06-20 09:56:45,225 - INFO - Event 20100106_035239: File sudah ada, skip.
2026-06-20 09:56:45,225 - INFO - Event 20100106_164500: File sudah ada, skip.
2026-06-20 09:56:45,226 - INFO - Event 20100106_202120: File sudah ada, skip.
2026-06-20 09:56:45,226 - INFO - Event 20100107_123517: File sudah ada, skip.
2026-06-20 09:56:45,226 - INFO - Event 20100107_173337: File sud

/opt/homebrew/Caskroom/miniforge/base/envs/tf-metal/lib/python3.11/site-packages/obspy/clients/fdsn/client.py:251: ObsPyDeprecationWarning: IRIS is now EarthScope, please consider changing the FDSN client short URL to 'EARTHSCOPE'.
  warnings.warn(msg, ObsPyDeprecationWarning)


2026-06-20 09:56:51,941 - INFO - Event 20100103_230900: Stasiun GE.SANI (jarak 3.52°)
2026-06-20 09:56:52,092 - INFO - Event 20100115_120847: Stasiun GE.SANI (jarak 1.90°)


...

2026-06-20 09:59:52,919 - INFO - Event 20100214_040249: Berhasil disimpan ke GE_BKNI_20100214_040249.mseed (trace: 3)
2026-06-20 09:59:53,873 - INFO - Event 20100216_232949: Stasiun GE.JAGI (jarak 4.53°)
2026-06-20 09:59:53,957 - INFO - Event 20100214_220917: Berhasil disimpan ke GE_GSI_20100214_220917.mseed (trace: 3)


/opt/homebrew/Caskroom/miniforge/base/envs/tf-metal/lib/python3.11/site-packages/obspy/clients/fdsn/client.py:251: ObsPyDeprecationWarning: IRIS is now EarthScope, please consider changing the FDSN client short URL to 'EARTHSCOPE'.
  warnings.warn(msg, ObsPyDeprecationWarning)


2026-06-20 09:59:54,984 - INFO - Event 20100216_111624: Stasiun GE.TNTI (jarak 3.73°)


/opt/homebrew/Caskroom/miniforge/base/envs/tf-metal/lib/python3.11/site-packages/obspy/clients/fdsn/client.py:251: ObsPyDeprecationWarning: IRIS is now EarthScope, please consider changing the FDSN client short URL to 'EARTHSCOPE'.
  warnings.warn(msg, ObsPyDeprecationWarning)


2026-06-20 09:59:55,524 - INFO - Event 20100215_143855: Berhasil disimpan ke GE_SAUI_20100215_143855.mseed (trace: 3)
